# 03 — Exp 1: Main Timing Benchmark (`When2Speak`)

1. Calibrate $(\lambda, \tau_1)$ strictly on `When2Speak-ValCalib` ($\le 200$ validation turns — **Invariant #8**) independently for `Luna` and `Terra`.
2. Evaluate `B0 Direct Chat`, `B1 Single Gate` (LLM & Jev), `B2 ProACT-Reimpl`, and `DIAL-Skill` (`Luna` & `Terra`).
3. Compute McNemar's test + Bootstrap 95% CIs and persist `metrics_summary.json` to Drive.
4. Default `MODE = 'pilot'` (400 turns); switch to `MODE = 'main'` (2,000 turns) for paper Table 1.

In [ ]:
# === CELL 0: Mount Google Drive & Configure Persistent Workspace ===
import os
from pathlib import Path

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or Path("/content").exists()
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = Path("/content/drive/MyDrive/when2speak_research")
else:
    DRIVE_ROOT = Path("./local_drive_mirror").resolve()

for sub in [
    "secrets",
    "config_cache",
    "datasets/pagemantle",
    "datasets/when2speak",
    "datasets/pagemantle_ext",
    "caches",
    "runs",
    "paper_artifacts/tables",
    "paper_artifacts/figures",
]:
    (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)

os.environ["DIAL_SKILL_DRIVE_ROOT"] = str(DRIVE_ROOT)
print("Persistent DRIVE_ROOT:", DRIVE_ROOT)


In [ ]:
# === CELL 1: Sync GitHub Repo (git clone / git pull) & Hot-Reload ===
REPO_URL = "https://github.com/BoldHu/multiagent-speak.git"  # Edit once with your repo URL
REPO_DIR = Path("/content/when2speak") if IN_COLAB else Path(".").resolve()

import importlib, sys, types

if IN_COLAB:
    if not REPO_DIR.exists():
        !git clone {REPO_URL} {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull --ff-only
    %cd {REPO_DIR}
    try:
        import typesafe_sdk, duckdb, typer, tenacity  # noqa: F401
        !pip install -q -e . --no-deps
    except ImportError:
        !pip install -q -e .

# Ensure `src/` is immediately importable in a fresh Colab kernel without restarting Python
_src_dir = str((REPO_DIR / "src").resolve())
if _src_dir not in sys.path:
    sys.path.insert(0, _src_dir)

# Colab Python 3.12+/3.13 removed stdlib `imp`, which older IPython autoreload still imports.
if "imp" not in sys.modules:
    try:
        import imp  # noqa: F401
    except ModuleNotFoundError:
        _imp_shim = types.ModuleType("imp")
        _imp_shim.reload = importlib.reload
        sys.modules["imp"] = _imp_shim

try:
    _ip = get_ipython()
    if _ip is not None:
        _ip.run_line_magic("load_ext", "autoreload")
        _ip.run_line_magic("autoreload", "2")
except Exception as _exc:
    print(f"Skipping autoreload ({_exc}); restart runtime after git pull if needed.")


In [ ]:
# === CELL 2: Bootstrap Secrets & Resolved Model Config (Never Prints Raw Keys) ===
import json
from dial_skill.utils.secrets import describe_secret_sources
from dial_skill.llm.preflight import load_resolved_models, resolve_model_for_role
from dial_skill.config import load_models_config, load_dial_skill_config, load_experiments_config

print("Secret lookup diagnostics:", json.dumps(describe_secret_sources(drive_root=DRIVE_ROOT), indent=2))
MODELS_CFG = load_models_config()
DIAL_CFG = load_dial_skill_config()
EXP_CFG = load_experiments_config()
RESOLVED_MODELS = load_resolved_models(drive_root=DRIVE_ROOT)
print("Resolved models:", RESOLVED_MODELS.model_dump() if RESOLVED_MODELS else "Not yet generated (run Notebook 00)")


## Step 1: Select Mode & Print Cost Estimate

In [ ]:
from dial_skill.eval.runners import estimate_run_cost_usd
from dial_skill.utils.checkpoint import save_run_metadata

MODE = "pilot"  # Change to "main" (2,000 turns) for full paper Table 1!
RUN_DIR = DRIVE_ROOT / "runs" / "main_when2speak" / MODE
est = estimate_run_cost_usd(mode=MODE, include_pagemantle=False)
print("Estimated Cost:", json.dumps(est, indent=2))
save_run_metadata(RUN_DIR, {"mode": MODE, "estimate": est})


## Step 2: Validation Calibration (`val_calib <= 200` turns, Zero Leakage — Independent for Luna & Terra)

In [ ]:
from dial_skill.benchmarks.when2speak import assert_no_calibration_leakage, load_when2speak_split
from dial_skill.core.pipeline import DialSkillAgent
from dial_skill.core.skill_store import SkillStore
from dial_skill.eval.runners import calibrate_when2speak_thresholds, run_when2speak_eval
from dial_skill.llm.jev_client import TriageClient
from dial_skill.llm.openai_client import OpenAIClient
from dial_skill.utils.cache import SQLiteAPICache

cache = SQLiteAPICache(DRIVE_ROOT / "caches" / "api_cache.sqlite")
oai = OpenAIClient(models_cfg=MODELS_CFG, cache=cache)
luna_model = resolve_model_for_role("luna", MODELS_CFG, RESOLVED_MODELS)
terra_model = resolve_model_for_role("terra", MODELS_CFG, RESOLVED_MODELS)
backend = RESOLVED_MODELS.triage_backend if RESOLVED_MODELS else MODELS_CFG.roles.triage_backend
triage_client = TriageClient(models_cfg=MODELS_CFG, cache=cache, openai_client=oai, luna_model=luna_model, backend=backend)

val_turns = load_when2speak_split("val_calib", local_dir=DRIVE_ROOT / "datasets" / "when2speak", seed=EXP_CFG["seed"])
eval_turns = load_when2speak_split(MODE, local_dir=DRIVE_ROOT / "datasets" / "when2speak", seed=EXP_CFG["seed"])
assert_no_calibration_leakage(val_turns, eval_turns, max_calib_turns=200)

# Calibrate Luna (gpt-4.1-mini) strictly on validation split <= 200 turns
calib_luna_agent = DialSkillAgent(
    triage_client=triage_client, openai_client=oai, gate_model=luna_model, executor_model=luna_model, cfg=DIAL_CFG
)
val_eps_luna = await run_when2speak_eval(
    agent=calib_luna_agent, turns=val_turns, output_jsonl=RUN_DIR / "val_calib_dial.jsonl", resume=True
)
calib_luna = calibrate_when2speak_thresholds(
    val_turns=val_turns,
    val_episodes=val_eps_luna,
    eval_turns=eval_turns,
    lambda_grid=EXP_CFG["when2speak"]["calibration"]["lambda_grid"],
    tau_dm_grid=EXP_CFG["when2speak"]["calibration"]["tau_dm_grid"],
    theta_floor=DIAL_CFG.gate.theta_floor,
    max_fir=0.11,
    max_mir=0.45,
)

# Calibrate Terra (gpt-4.1) independently on the exact same validation split <= 200 turns
calib_terra_agent = DialSkillAgent(
    triage_client=triage_client, openai_client=oai, gate_model=terra_model, executor_model=terra_model, cfg=DIAL_CFG
)
val_eps_terra = await run_when2speak_eval(
    agent=calib_terra_agent, turns=val_turns, output_jsonl=RUN_DIR / "val_calib_dial_terra.jsonl", resume=True
)
calib_terra = calibrate_when2speak_thresholds(
    val_turns=val_turns,
    val_episodes=val_eps_terra,
    eval_turns=eval_turns,
    lambda_grid=EXP_CFG["when2speak"]["calibration"]["lambda_grid"],
    tau_dm_grid=EXP_CFG["when2speak"]["calibration"]["tau_dm_grid"],
    theta_floor=DIAL_CFG.gate.theta_floor,
    max_fir=0.11,
    max_mir=0.45,
)

calib = calib_luna
calib_bundle = {"luna": calib_luna, "terra": calib_terra, **calib_luna}
print("Calibrated scalars (Luna):", json.dumps(calib_luna, indent=2))
print("Calibrated scalars (Terra):", json.dumps(calib_terra, indent=2))
(RUN_DIR / "calibration.json").write_text(json.dumps(calib_bundle, indent=2), encoding="utf-8")


## Step 3: Run Baselines (`B0`, `B1`, `B2 ProACT`) & `DIAL-Skill` (`Luna` & `Terra`)

In [ ]:
from dial_skill.baselines.direct_chat import DirectChatBaseline
from dial_skill.baselines.proact import ProACTBaseline
from dial_skill.baselines.single_gate import SingleGateBaseline
from dial_skill.eval.metrics import compute_when2speak_metrics, mcnemar_test

tuned_cfg_luna = load_dial_skill_config(
    overrides={"gate": {"lambda_disruption": calib_luna["best_lambda"], "tau_dm": calib_luna["best_tau_dm"]}}
)
tuned_cfg_terra = load_dial_skill_config(
    overrides={"gate": {"lambda_disruption": calib_terra["best_lambda"], "tau_dm": calib_terra["best_tau_dm"]}}
)

methods = {
    "B0_direct_luna": DirectChatBaseline(openai_client=oai, model=luna_model),
    "B1_single_llm_luna": SingleGateBaseline(openai_client=oai, gate_model=luna_model, gate_source="llm"),
    "B1_single_jev": SingleGateBaseline(openai_client=oai, gate_model=luna_model, triage_client=triage_client, gate_source="jev"),
    "B2_proact_luna": ProACTBaseline(openai_client=oai, gate_model=luna_model),
    "DIAL_Skill_luna": DialSkillAgent(
        triage_client=triage_client, openai_client=oai, gate_model=luna_model, executor_model=luna_model, cfg=tuned_cfg_luna, method_name="DIAL_Skill_luna"
    ),
    "DIAL_Skill_terra": DialSkillAgent(
        triage_client=triage_client, openai_client=oai, gate_model=terra_model, executor_model=terra_model, cfg=tuned_cfg_terra, method_name="DIAL_Skill_terra"
    ),
}

y_true = [t.label_speak for t in eval_turns]
results_summary = {}
preds_by_method = {}

for name, agent in methods.items():
    eps = await run_when2speak_eval(
        agent=agent, turns=eval_turns, output_jsonl=RUN_DIR / f"preds_{name}.jsonl", resume=True, generate_text=False
    )
    preds_by_method[name] = [e.gate.binary_speak for e in eps]
    results_summary[name] = compute_when2speak_metrics(y_true, eps)
    m = results_summary[name]
    print(f"[{name:20s}] Macro F1={m['macro_f1']:.3f} | Speak F1={m['speak_f1']:.3f} | FIR={m['fir']:.3f} | MIR={m['mir']:.3f} | AUROC={m['auroc']:.3f} | TokRed={m['token_reduction_pct']:.1f}%")

sig_vs_direct = mcnemar_test(y_true, preds_by_method["DIAL_Skill_luna"], preds_by_method["B0_direct_luna"])
sig_vs_proact = mcnemar_test(y_true, preds_by_method["DIAL_Skill_luna"], preds_by_method["B2_proact_luna"])
sig_vs_b1_jev = mcnemar_test(y_true, preds_by_method["DIAL_Skill_luna"], preds_by_method["B1_single_jev"])
sig_terra_vs_direct = mcnemar_test(y_true, preds_by_method["DIAL_Skill_terra"], preds_by_method["B0_direct_luna"])

print("McNemar DIAL_Luna vs Direct:", sig_vs_direct)
print("McNemar DIAL_Luna vs ProACT:", sig_vs_proact)
print("McNemar DIAL_Luna vs B1_Jev:", sig_vs_b1_jev)
print(f"Total run spend: OpenAI=${oai.usage.total_cost_usd:.4f}, Jev=${triage_client.total_cost_usd:.4f}")

summary_payload = {
    "mode": MODE,
    "calibration": calib_bundle,
    "metrics": results_summary,
    "mcnemar_dial_vs_direct": sig_vs_direct,
    "mcnemar_dial_vs_proact": sig_vs_proact,
    "mcnemar_dial_vs_b1_jev": sig_vs_b1_jev,
    "mcnemar_terra_vs_direct": sig_terra_vs_direct,
    "triage_meta": triage_client.get_run_metadata(),
}
(RUN_DIR / "metrics_summary.json").write_text(json.dumps(summary_payload, indent=2), encoding="utf-8")
print("Saved metrics_summary.json to Drive.")
